In [ ]:
!pip install transformers torch emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SENTIMENT_MODEL_PATH = "/content/drive/MyDrive/review_analysis/models/sentiment_model"
DATA_PATH = "/content/drive/MyDrive/review_analysis/data"

import os
os.makedirs(SENTIMENT_MODEL_PATH, exist_ok=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import torch
import re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Reproducibility seeds
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import emoji
def preprocess_text(text):
    text = str(text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s.!?':_-]", '', text)
    text = ' '.join(text.split())
    return text

# Load data
df = pd.read_csv(f'{DATA_PATH}/combined_reviews_dataset.csv')
df = df.rename(columns={'review': 'text', 'class': 'label'})

initial_count = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_count - len(df)} duplicate rows.")

df.dropna(subset=['text', 'label'], inplace=True)

# Validate binary labels
unique_labels = sorted(df['label'].unique())
assert list(unique_labels) == [0, 1], f"Expected binary labels [0, 1], got {unique_labels}"

df['Cleaned_Review'] = df['text'].apply(preprocess_text)

print(f"Total samples: {len(df)}")
print(f"Class distribution: {dict(df['label'].value_counts())}")

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Get data
reviews = df['Cleaned_Review'].tolist()
labels = df['label'].tolist()

# 3-way stratified split: train (80%) -> val (10%) / test (10%)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    reviews, labels, test_size=0.2, random_state=42, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

# Create datasets
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)
test_dataset = ReviewDataset(test_texts, test_labels, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)

print("✓ Model loaded!")

In [ ]:
def train_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            total_loss += outputs.loss.item()

            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), correct / total

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 4
best_accuracy = 0
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Starting training...")
print("=" * 50)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_loss, val_acc = evaluate(model, val_loader)

    scheduler.step()

    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"  LR:         {scheduler.get_last_lr()[0]:.2e}")

    # Save best model to Drive
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), f"{SENTIMENT_MODEL_PATH}/best_model.pt")
        print("  ✓ Best model saved!")

print("\n" + "=" * 50)
print(f"Training complete! Best val accuracy: {best_accuracy:.4f}")

In [ ]:
# Load best weights from Drive
model.load_state_dict(torch.load(f"{SENTIMENT_MODEL_PATH}/best_model.pt", weights_only=True))
model.eval()

# Final evaluation on held-out test set (never used for model selection)
test_loss, test_acc = evaluate(model, test_loader)
print(f"\n--- Final Test Evaluation ---")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
model.save_pretrained(SENTIMENT_MODEL_PATH)
tokenizer.save_pretrained(SENTIMENT_MODEL_PATH)
print(f"✓ Model saved to: {SENTIMENT_MODEL_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to: /content/drive/MyDrive/review_analysis/models/sentiment_model


In [ ]:
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment of a single review. Caller must set model.eval() beforehand."""
    cleaned_text = preprocess_text(text)

    inputs = tokenizer(
        cleaned_text,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0][predicted_class].item()

    return {
        "text": text,
        "label": "Positive" if predicted_class == 1 else "Negative",
        "class": predicted_class,
        "confidence": confidence
    }


def filter_negative_reviews(reviews, model, tokenizer):
    """Filter only negative reviews."""
    model.eval()
    negative_reviews = []

    for review in reviews:
        result = predict_sentiment(review, model, tokenizer)
        if result["class"] == 0:
            negative_reviews.append(review)

    return negative_reviews

In [ ]:
print("\n--- Testing Inference ---")

test_reviews = [
    "Amazing event! Everything was perfect!",
    "Terrible experience. Rude staff and long lines.",
    "The workshop was okay, nothing special.",
    "Worst conference ever. Waste of money."
]

for review in test_reviews:
    result = predict_sentiment(review, model, tokenizer)
    print(f"{result['label']} ({result['confidence']:.2f}): {review}")

# Filter negatives
negative_only = filter_negative_reviews(test_reviews, model, tokenizer)
print(f"\nFound {len(negative_only)} negative reviews:")
for r in negative_only:
    print(f"  - {r}")


--- Testing Inference ---
Positive (1.00): Amazing event! Everything was perfect!
Negative (1.00): Terrible experience. Rude staff and long lines.
Negative (1.00): The workshop was okay, nothing special.
Negative (1.00): Worst conference ever. Waste of money.

Found 3 negative reviews:
  - Terrible experience. Rude staff and long lines.
  - The workshop was okay, nothing special.
  - Worst conference ever. Waste of money.


In [ ]:
# Install
!pip install -q huggingface_hub

# Login
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
HF_USERNAME = "MHKaungPyae"

model.push_to_hub(f"{HF_USERNAME}/sentiment-model")
tokenizer.push_to_hub(f"{HF_USERNAME}/sentiment-model")
print("✓ Sentiment model uploaded!")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...aab_gvo/model.safetensors:   0%|          |  575kB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Sentiment model uploaded!


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions on test set
model.eval()
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Classification report
print(classification_report(all_labels, all_preds, target_names=["Negative", "Positive"]))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"],
            linewidths=1, linecolor='gray',
            annot_kws={"size": 16})
plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
plt.ylabel('True Label', fontsize=13, fontweight='bold')
plt.title('Confusion Matrix — DistilBERT Sentiment Model', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved as confusion_matrix.png")